# MOLM-ST Mutation-Holdout Evaluation

This notebook runs the **original MOLM-ST architecture/trainer** under the current mutation-holdout protocol so that the independent same-loss control can be compared directly with Standard-MOLM and the other models.

### Scientific purpose

The extended mutation analysis includes Standard-MOLM, Routed-MOLM, NN, and LDA. This notebook adds a dedicated MOLM-ST run using the original plain MOLM-ST implementation rather than re-labeling the independent arm from the separate A--H component suite.

### Locked protocol

- Repository commit: `c5923984f0d5176977edb4a4ffd8fc5f98536043`
- Original `phase0_config.train_molm_st()` implementation
- Five optimization seeds: `42, 123, 456, 789, 2024`
- Five representations: `onehot`, `mean_esm2`, `mean_fusion`, `site_esm2`, `site_fusion`
- Primary top-residue mutation holdouts at the same eight CDR sites
- Optional secondary wild-type holdouts
- Two independent full MOLM-ST models per seed / representation / holdout: target-binding proxy and OVA-binding proxy
- Same focal + ranking + gap objective as Standard-MOLM
- 25 epochs, batch size 64, AdamW learning rate `5e-5`
- T4 x2 parallel execution with resumable jobs

The notebook writes per-seed predictions, site-level summaries, optional paired Standard-MOLM vs MOLM-ST site-block tests when a compatible existing result bundle is mounted, a reproducibility manifest, and a final result ZIP.

> **Important:** Arm F from the A--H routed ablation is not used here. This notebook uses the original plain MOLM-ST implementation.


In [ ]:

from pathlib import Path
import os, sys, json, time, shutil, subprocess, platform, hashlib, re, math, zipfile
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/DigantaX/molm-pipeline.git"
PINNED_COMMIT = "c5923984f0d5176977edb4a4ffd8fc5f98536043"

SEEDS = [42, 123, 456, 789, 2024]
FEATURE_TYPES = ["onehot", "mean_esm2", "mean_fusion", "site_esm2", "site_fusion"]
FEATURE_DIMS = {
    "onehot": 2300,
    "mean_esm2": 320,
    "mean_fusion": 2620,
    "site_esm2": 2560,
    "site_fusion": 4860,
}

EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 5e-5

# Primary manuscript experiment = top-residue holdouts only.
# Turn this on only if you also want the secondary WT holdouts.
RUN_WILDTYPE_SECONDARY = False

REQUIRE_T4_X2 = True
RESUME = True
ALLOW_SHAPE_VERIFIED_NPY_REUSE = True

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/molm_st_mutation_holdout")
REPO = WORK_ROOT / "molm-pipeline"
FEATURE_DIR = WORK_ROOT / "features"
JOBS_DIR = WORK_ROOT / "jobs"
LOG_DIR = WORK_ROOT / "logs"
ANALYSIS_DIR = WORK_ROOT / "analysis"
CODE_DIR = WORK_ROOT / "code"
EXISTING_RESULTS_DIR = WORK_ROOT / "existing_results"

for d in [WORK_ROOT, FEATURE_DIR, JOBS_DIR, LOG_DIR, ANALYSIS_DIR, CODE_DIR, EXISTING_RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Seeds:", SEEDS)
print("Features:", FEATURE_TYPES)
print("Primary holdouts only:", not RUN_WILDTYPE_SECONDARY)



## 1. Runtime preflight and pinned repository

Use Kaggle **GPU T4 x2**. Internet may be turned on for the initial clone/install; if you mount a repository snapshot and compatible ESM cache, the notebook can reuse them.


In [ ]:

import importlib.metadata
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPUs:", [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())])

if REQUIRE_T4_X2 and torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"Please select Kaggle accelerator 'GPU T4 x2'. Found {torch.cuda.device_count()} GPU(s)."
    )

try:
    esm_version = importlib.metadata.version("fair-esm")
except importlib.metadata.PackageNotFoundError:
    esm_version = None

if esm_version != "2.0.0":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fair-esm==2.0.0"],
        check=True
    )
print("fair-esm:", importlib.metadata.version("fair-esm"))

# Prefer a mounted repository snapshot when available.
repo_candidates = []
if INPUT_ROOT.exists():
    for p in INPUT_ROOT.rglob("phase0_config.py"):
        parent = p.parent
        if (parent / "data" / "emi_binding.csv").exists():
            repo_candidates.append(parent)

if REPO.exists() and not RESUME:
    shutil.rmtree(REPO)

if not REPO.exists():
    if repo_candidates:
        shutil.copytree(repo_candidates[0], REPO)
        print("Using mounted repository snapshot:", repo_candidates[0])
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)

subprocess.run(["git", "-C", str(REPO), "checkout", "-q", PINNED_COMMIT], check=True)
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip()
assert commit == PINNED_COMMIT, (commit, PINNED_COMMIT)

# The seed must be supplied before importing phase0_config inside each worker.
sys.path.insert(0, str(REPO))
import phase0_config as pc

cfg = pc.config

# Scientific audit: fail loudly if the pinned implementation differs.
assert list(cfg.SHARED_DIMS) == [256, 128]
assert list(cfg.TOWER_DIMS) == [64, 32]
assert int(cfg.LATENT_DIM) == 16
assert float(cfg.DROPOUT_RATE) == 0.2
assert float(cfg.RANKING_WEIGHT_AFF) == 0.3
assert float(cfg.RANKING_WEIGHT_SPEC) == 0.6
assert float(cfg.GAP_WEIGHT_AFF) == 0.2
assert float(cfg.GAP_WEIGHT_SPEC) == 0.6
assert float(cfg.RANKING_MARGIN) == 0.3
assert float(cfg.GAP_MARGIN) == 0.2
assert list(cfg.MUTATION_SITES) == [32, 49, 54, 55, 56, 98, 100, 103]
assert list(cfg.MUTATION_SITES_KABAT) == [33, 50, 54, 55, 56, 95, 97, 102]
assert hasattr(pc, "train_molm_st")
assert hasattr(pc, "MOLMSingleTaskTrainer")

print("Pinned repository:", commit)
print("Original MOLM-ST implementation found: phase0_config.train_molm_st")



## 2. EMI labels and exact five representations

OneHot is generated using the pinned repository implementation. For ESM-2, compatible cached EMI matrices under `/kaggle/input` are reused after strict dimension checks. Missing matrices are computed with `esm2_t6_8M_UR50D`, layer 6.

- Mean-ESM2: whole-VH mean pooling, 320D
- Site-ESM2: concatenated residue embeddings at the eight designed CDR positions, 2560D


In [ ]:

import gc
import esm

def resolve_column(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    raise KeyError(f"None of {candidates} found. Columns={list(frame.columns)}")

emi_binding = pd.read_csv(REPO / "data" / "emi_binding.csv", index_col=0)
emi_sequences = emi_binding.index.astype(str).to_numpy()

lengths = np.asarray([len(s) for s in emi_sequences])
assert np.all(lengths == 115), sorted(set(lengths.tolist()))

aff_col = resolve_column(emi_binding, ["ANT Binding", "ANT", "Affinity", "affinity"])
ova_col = resolve_column(emi_binding, ["OVA Binding", "OVA", "PSY", "Specificity", "specificity"])

y_aff = (emi_binding[aff_col].to_numpy() > 0).astype(np.float32)
y_ova = (emi_binding[ova_col].to_numpy() > 0).astype(np.float32)

aff_pos_weight = float((1.0 - y_aff.mean()) / max(float(y_aff.mean()), 1e-6))
ova_pos_weight = 1.0

np.save(FEATURE_DIR / "y_aff.npy", y_aff)
np.save(FEATURE_DIR / "y_ova.npy", y_ova)
np.save(FEATURE_DIR / "emi_sequences.npy", emi_sequences.astype("U115"))
np.save(FEATURE_DIR / "pos_weights.npy", np.asarray([aff_pos_weight, ova_pos_weight], np.float32))

print("EMI n:", len(emi_sequences))
print("Target positive fraction:", float(y_aff.mean()))
print("Target positive weight:", aff_pos_weight)
print("OVA positive fraction:", float(y_ova.mean()))

def fpath(feature):
    return FEATURE_DIR / f"emi_{feature}.npy"

# OneHot from the pinned repository.
if not (RESUME and fpath("onehot").exists()):
    onehot = pc.generate_onehot(emi_binding).to_numpy(np.float32)
    assert onehot.shape == (len(emi_sequences), 2300)
    np.save(fpath("onehot"), onehot)
else:
    onehot = np.load(fpath("onehot"), mmap_mode="r")
print("OneHot:", onehot.shape)

EXPECTED = {"mean_esm2": 320, "site_esm2": 2560}

def candidate_kind(path):
    s = path.stem.lower()
    if "site" in s and ("esm" in s or "embedding" in s):
        return "site_esm2"
    if ("mean" in s and ("esm" in s or "embedding" in s)) or re.search(r"(^|[_\-.])esm2?([_\-.]|$)", s):
        return "mean_esm2"
    return None

def candidate_is_emi(path):
    s = str(path).lower()
    return (
        re.search(r"(^|[/_\-.])emi([/_\-.]|$)", s) is not None
        or "emibetuzumab" in s
    )

def load_candidate(path, kind):
    n, d = len(emi_sequences), EXPECTED[kind]
    try:
        if path.suffix.lower() == ".npy":
            if not ALLOW_SHAPE_VERIFIED_NPY_REUSE:
                return None
            arr = np.load(path, mmap_mode="r")
            if arr.shape == (n, d):
                return np.asarray(arr, np.float32)
            return None

        name = path.name.lower()
        if name.endswith(".csv") or name.endswith(".csv.gz") or name.endswith(".tsv"):
            sep = "\t" if name.endswith(".tsv") else ","
            df = pd.read_csv(path, sep=sep)
            seq_col = next((c for c in df.columns if c.lower() in {"sequence","sequence_id","seq","vh_sequence"}), None)
            if seq_col is not None:
                idx = pd.Index(df[seq_col].astype(str))
                loc = idx.get_indexer(pd.Index(emi_sequences))
                if (loc < 0).any():
                    return None
                numeric = df.drop(columns=[seq_col]).select_dtypes(include=[np.number])
                if numeric.shape[1] != d:
                    return None
                return numeric.to_numpy(np.float32)[loc]
            numeric = df.select_dtypes(include=[np.number])
            if numeric.shape == (n, d):
                return numeric.to_numpy(np.float32)
    except Exception:
        return None
    return None

# Reuse compatible ESM arrays if mounted.
for kind in ["mean_esm2", "site_esm2"]:
    dst = fpath(kind)
    if RESUME and dst.exists():
        arr = np.load(dst, mmap_mode="r")
        if arr.shape == (len(emi_sequences), EXPECTED[kind]):
            print("Working cache:", kind, arr.shape)
            continue

    accepted = None
    if INPUT_ROOT.exists():
        candidates = []
        for ext in ("*.npy", "*.csv", "*.csv.gz", "*.tsv"):
            candidates.extend(INPUT_ROOT.rglob(ext))
        for p in sorted(set(candidates)):
            if not candidate_is_emi(p) or candidate_kind(p) != kind:
                continue
            mat = load_candidate(p, kind)
            if mat is not None:
                accepted = (p, mat)
                break

    if accepted is not None:
        src, mat = accepted
        np.save(dst, mat.astype(np.float32, copy=False))
        print("Reused:", kind, "<-", src)
    else:
        print("Missing:", kind, "-> will compute")

missing_mean = not fpath("mean_esm2").exists()
missing_site = not fpath("site_esm2").exists()

if missing_mean or missing_site:
    print("Computing missing ESM features...")
    model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
    model.eval()
    device = torch.device("cuda:0")
    model = model.to(device)
    batch_converter = alphabet.get_batch_converter()

    seq_sites = list(map(int, cfg.MUTATION_SITES))
    token_sites = [i + 1 for i in seq_sites]  # BOS token offset

    means = [] if missing_mean else None
    sites = [] if missing_site else None
    data = [(f"s{i}", str(s)) for i, s in enumerate(emi_sequences)]

    with torch.no_grad():
        for start in range(0, len(data), 64):
            batch = data[start:start+64]
            _, _, tokens = batch_converter(batch)
            tokens = tokens.to(device)
            reps = model(tokens, repr_layers=[6], return_contacts=False)["representations"][6]

            for j, (_, seq) in enumerate(batch):
                L = len(seq)
                if missing_mean:
                    means.append(reps[j, 1:L+1, :].mean(0).float().cpu().numpy())
                if missing_site:
                    sites.append(reps[j, token_sites, :].reshape(-1).float().cpu().numpy())

            if start == 0 or (start // 64 + 1) % 10 == 0:
                print(f"  {min(start+64, len(data))}/{len(data)}")

    if missing_mean:
        arr = np.stack(means).astype(np.float32)
        assert arr.shape == (len(emi_sequences), 320)
        np.save(fpath("mean_esm2"), arr)
    if missing_site:
        arr = np.stack(sites).astype(np.float32)
        assert arr.shape == (len(emi_sequences), 2560)
        np.save(fpath("site_esm2"), arr)

    del model
    gc.collect()
    torch.cuda.empty_cache()

# Fusion features.
oh = np.load(fpath("onehot"), mmap_mode="r")
me = np.load(fpath("mean_esm2"), mmap_mode="r")
se = np.load(fpath("site_esm2"), mmap_mode="r")

if not (RESUME and fpath("mean_fusion").exists()):
    np.save(fpath("mean_fusion"), np.concatenate([oh, me], axis=1).astype(np.float32))
if not (RESUME and fpath("site_fusion").exists()):
    np.save(fpath("site_fusion"), np.concatenate([oh, se], axis=1).astype(np.float32))

print("\nFinal feature audit:")
for f in FEATURE_TYPES:
    arr = np.load(fpath(f), mmap_mode="r")
    assert arr.shape == (len(emi_sequences), FEATURE_DIMS[f]), (f, arr.shape)
    print(f"  {f:12s}", arr.shape)



## 3. Lock the mutation holdouts

The primary analysis withholds every sequence carrying the predefined top-performing residue at one CDR site. Training contains **zero** sequences carrying that residue at that site.

The optional secondary analysis uses the corresponding wild-type residue in the same way.


In [ ]:

holdouts = []

modes = [("top", list(cfg.TOP_RESIDUES), "primary")]
if RUN_WILDTYPE_SECONDARY:
    modes.append(("wildtype", list(cfg.WILDTYPE_RESIDUES), "secondary"))

for holdout_type, residues, analysis_role in modes:
    for ordinal, (site_idx, kabat, residue) in enumerate(zip(
        cfg.MUTATION_SITES,
        cfg.MUTATION_SITES_KABAT,
        residues
    )):
        test_idx = np.asarray(
            [i for i, s in enumerate(emi_sequences) if s[int(site_idx)] == str(residue)],
            dtype=int
        )
        train_idx = np.asarray(
            [i for i, s in enumerate(emi_sequences) if s[int(site_idx)] != str(residue)],
            dtype=int
        )

        assert len(test_idx) > 0
        assert len(train_idx) + len(test_idx) == len(emi_sequences)
        assert all(emi_sequences[i][int(site_idx)] != str(residue) for i in train_idx)
        assert all(emi_sequences[i][int(site_idx)] == str(residue) for i in test_idx)

        holdouts.append({
            "holdout_id": f"{holdout_type}_kabat{kabat}_{residue}",
            "holdout_ordinal": ordinal,
            "holdout_type": holdout_type,
            "analysis_role": analysis_role,
            "python_index": int(site_idx),
            "kabat_site": int(kabat),
            "heldout_residue": str(residue),
            "train_idx": train_idx.tolist(),
            "test_idx": test_idx.tolist(),
            "train_n": int(len(train_idx)),
            "test_n": int(len(test_idx)),
        })

HOLDOUT_PATH = WORK_ROOT / "MOLM_ST_MUTATION_HOLDOUTS_LOCKED.json"
HOLDOUT_PATH.write_text(json.dumps(holdouts, indent=2))

audit = pd.DataFrame([
    {k: v for k, v in h.items() if k not in {"train_idx", "test_idx"}}
    for h in holdouts
])
display(audit)

assert audit[audit.analysis_role == "primary"].shape[0] == 8
print("Locked primary holdouts:", int((audit.analysis_role == "primary").sum()))
print("Total holdouts:", len(audit))
print("Holdout manifest:", HOLDOUT_PATH)



## 4. Exact MOLM-ST mutation worker

Each `(seed, representation)` worker loops over the locked holdouts and calls the pinned repository's **original** `train_molm_st()` twice:

- `task="affinity"`, seed offset `+500`
- `task="specificity"`, seed offset `+600`

Each model is a complete independent `DiagnosticMOLM` instance and receives only its selected task's focal + ranking + gap loss.


In [ ]:
WORKER_PATH = CODE_DIR / 'molm_st_mutation_worker.py'
WORKER_SOURCE = '\nfrom __future__ import annotations\nimport os, sys, json, gc, time\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import (\n    accuracy_score, balanced_accuracy_score, matthews_corrcoef,\n    roc_auc_score, average_precision_score\n)\n\nrepo = Path(os.environ["MOLM_REPO"])\nfeature_dir = Path(os.environ["MOLM_FEATURE_DIR"])\noutput_dir = Path(os.environ["MOLM_JOB_OUTPUT"])\nholdout_path = Path(os.environ["MOLM_HOLDOUT_PATH"])\nfeature = os.environ["MOLM_FEATURE"]\nseed = int(os.environ["MOLM_SEED"])\n\noutput_dir.mkdir(parents=True, exist_ok=True)\nsys.path.insert(0, str(repo))\n\n# IMPORTANT: environment seed is set before this import.\nimport phase0_config as pc\n\ncfg = pc.config\ncfg.EPOCHS = int(os.environ.get("MOLM_EPOCHS", "25"))\ncfg.BATCH_SIZE = int(os.environ.get("MOLM_BATCH_SIZE", "64"))\ncfg.LEARNING_RATE = float(os.environ.get("MOLM_LEARNING_RATE", "5e-5"))\ncfg.PARETO_LOSS = False\ncfg.ADVERSARIAL_WEIGHT = 0.0\ncfg.ORTHO_WEIGHT = 0.0\n\nX = np.load(feature_dir / f"emi_{feature}.npy").astype(np.float32, copy=False)\ny_aff = np.load(feature_dir / "y_aff.npy").astype(np.float32)\ny_ova = np.load(feature_dir / "y_ova.npy").astype(np.float32)\nseqs = np.load(feature_dir / "emi_sequences.npy").astype(str)\nweights = np.load(feature_dir / "pos_weights.npy").astype(float)\naff_pw, ova_pw = float(weights[0]), float(weights[1])\nholdouts = json.loads(holdout_path.read_text())\n\ndef predict_scores(model, Xte, task):\n    device = next(model.parameters()).device\n    model.eval()\n    out = []\n    with torch.no_grad():\n        for start in range(0, len(Xte), 1024):\n            xb = torch.as_tensor(\n                Xte[start:start+1024],\n                dtype=torch.float32,\n                device=device\n            )\n            pred = model(xb, training=False)\n            key = "logit_aff" if task == "affinity" else "logit_spec"\n            out.append(pred[key].detach().cpu().numpy().reshape(-1))\n    return np.concatenate(out)\n\ndef metrics(y, score):\n    y = np.asarray(y, int)\n    score = np.asarray(score, float)\n    pred = (score >= 0).astype(int)\n    result = {\n        "n": int(len(y)),\n        "accuracy": float(accuracy_score(y, pred)),\n        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),\n        "mcc": float(matthews_corrcoef(y, pred)),\n    }\n    if len(np.unique(y)) == 2:\n        result["auroc"] = float(roc_auc_score(y, score))\n        result["auprc"] = float(average_precision_score(y, score))\n    else:\n        result["auroc"] = np.nan\n        result["auprc"] = np.nan\n    return result, pred\n\nall_raw = []\nall_metrics = []\nall_params = []\n\nfor h in holdouts:\n    root = output_dir / h["holdout_id"]\n    root.mkdir(parents=True, exist_ok=True)\n    done = root / "DONE.json"\n\n    expected = {\n        "seed": seed,\n        "feature": feature,\n        "holdout_id": h["holdout_id"],\n        "repository_commit": os.environ["MOLM_COMMIT"],\n    }\n\n    if done.exists():\n        try:\n            old = json.loads(done.read_text())\n            if all(old.get(k) == v for k, v in expected.items()):\n                all_raw.append(pd.read_csv(root / "raw_predictions.csv.gz"))\n                all_metrics.append(pd.read_csv(root / "metrics.csv"))\n                all_params.append(pd.read_csv(root / "parameter_counts.csv"))\n                continue\n        except Exception:\n            pass\n\n    tr = np.asarray(h["train_idx"], dtype=int)\n    te = np.asarray(h["test_idx"], dtype=int)\n\n    # Original plain MOLM-ST: two independent full instances.\n    target_model = pc.train_molm_st(\n        X[tr], y_aff[tr], y_ova[tr],\n        task="affinity",\n        config_obj=cfg,\n        aff_pos_weight=aff_pw,\n        spec_pos_weight=ova_pw,\n        seed_offset=500,\n        verbose=0,\n    )\n    target_score = predict_scores(target_model, X[te], "affinity")\n\n    ova_model = pc.train_molm_st(\n        X[tr], y_aff[tr], y_ova[tr],\n        task="specificity",\n        config_obj=cfg,\n        aff_pos_weight=aff_pw,\n        spec_pos_weight=ova_pw,\n        seed_offset=600,\n        verbose=0,\n    )\n    ova_score = predict_scores(ova_model, X[te], "specificity")\n\n    raw_rows = []\n    metric_rows = []\n\n    for task, truth, score in [\n        ("affinity", y_aff[te], target_score),\n        ("ova", y_ova[te], ova_score),\n    ]:\n        met, pred = metrics(truth, score)\n        metric_rows.append({\n            "seed": seed,\n            "seed_protocol": "optimization",\n            "feature": feature,\n            "model": "MOLM-ST",\n            "task": task,\n            **{k: v for k, v in h.items() if k not in {"train_idx", "test_idx"}},\n            **met,\n        })\n\n        for j, gid in enumerate(te):\n            raw_rows.append({\n                "seed": seed,\n                "seed_protocol": "optimization",\n                "feature": feature,\n                "model": "MOLM-ST",\n                "task": task,\n                **{k: v for k, v in h.items() if k not in {"train_idx", "test_idx"}},\n                "global_row_id": int(gid),\n                "sequence_id": str(seqs[gid]),\n                "y_true": int(truth[j]),\n                "score": float(score[j]),\n                "y_pred": int(pred[j]),\n            })\n\n    n_target = sum(p.numel() for p in target_model.parameters() if p.requires_grad)\n    n_ova = sum(p.numel() for p in ova_model.parameters() if p.requires_grad)\n\n    raw_df = pd.DataFrame(raw_rows)\n    met_df = pd.DataFrame(metric_rows)\n    par_df = pd.DataFrame([{\n        "seed": seed,\n        "feature": feature,\n        "holdout_id": h["holdout_id"],\n        "model": "MOLM-ST",\n        "target_instance_parameters": int(n_target),\n        "ova_instance_parameters": int(n_ova),\n        "total_two_instance_parameters": int(n_target + n_ova),\n    }])\n\n    raw_df.to_csv(root / "raw_predictions.csv.gz", index=False, compression="gzip")\n    met_df.to_csv(root / "metrics.csv", index=False)\n    par_df.to_csv(root / "parameter_counts.csv", index=False)\n    done.write_text(json.dumps(expected, indent=2))\n\n    all_raw.append(raw_df)\n    all_metrics.append(met_df)\n    all_params.append(par_df)\n\n    del target_model, ova_model\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\npd.concat(all_raw, ignore_index=True).to_csv(\n    output_dir / "raw_predictions.csv.gz", index=False, compression="gzip"\n)\npd.concat(all_metrics, ignore_index=True).to_csv(\n    output_dir / "metrics.csv", index=False\n)\npd.concat(all_params, ignore_index=True).to_csv(\n    output_dir / "parameter_counts.csv", index=False\n)\n\n(output_dir / "JOB_DONE.json").write_text(json.dumps({\n    "status": "ok",\n    "seed": seed,\n    "feature": feature,\n    "model": "MOLM-ST",\n    "repository_commit": os.environ["MOLM_COMMIT"],\n}, indent=2))\n'
compile(WORKER_SOURCE, str(WORKER_PATH), 'exec')
WORKER_PATH.write_text(WORKER_SOURCE, encoding='utf-8')
print('Worker:', WORKER_PATH)
print('Worker SHA256:', hashlib.sha256(WORKER_SOURCE.encode()).hexdigest())



## 5. Run the 25 seed × representation jobs on T4×2

Two workers run concurrently, one on each GPU. Each worker is resumable at the individual holdout level.


In [ ]:

import queue, threading

def job_root(seed, feature):
    return JOBS_DIR / f"seed_{seed}" / feature

def complete(seed, feature):
    root = job_root(seed, feature)
    return (root / "JOB_DONE.json").exists() and (root / "metrics.csv").exists()

def run_job(job, gpu):
    seed, feature = job
    root = job_root(seed, feature)
    root.mkdir(parents=True, exist_ok=True)

    if RESUME and complete(seed, feature):
        return {
            "seed": seed, "feature": feature, "gpu": gpu,
            "status": "skipped_verified"
        }

    log_path = LOG_DIR / f"seed_{seed}_{feature}.log"
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "MOLM_REPO": str(REPO),
        "MOLM_FEATURE_DIR": str(FEATURE_DIR),
        "MOLM_JOB_OUTPUT": str(root),
        "MOLM_HOLDOUT_PATH": str(HOLDOUT_PATH),
        "MOLM_FEATURE": feature,
        "MOLM_SEED": str(seed),
        "MOLM_EPOCHS": str(EPOCHS),
        "MOLM_BATCH_SIZE": str(BATCH_SIZE),
        "MOLM_LEARNING_RATE": str(LEARNING_RATE),
        "MOLM_COMMIT": PINNED_COMMIT,
        "PYTHONPATH": str(REPO),
    })

    print(f"[GPU {gpu}] START seed={seed} feature={feature}", flush=True)
    t0 = time.time()
    with log_path.open("w") as log:
        p = subprocess.run(
            [sys.executable, "-u", str(WORKER_PATH)],
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

    if p.returncode != 0:
        tail = "\n".join(log_path.read_text(errors="replace").splitlines()[-120:])
        raise RuntimeError(
            f"Job failed seed={seed}, feature={feature}, gpu={gpu}\n{tail}"
        )

    elapsed = time.time() - t0
    print(f"[GPU {gpu}] DONE seed={seed} feature={feature} {elapsed/60:.1f} min", flush=True)
    return {
        "seed": seed, "feature": feature, "gpu": gpu,
        "status": "ok", "runtime_seconds": elapsed
    }

jobs = [(seed, feature) for seed in SEEDS for feature in FEATURE_TYPES]
q = queue.Queue()
for job in jobs:
    q.put(job)

records = []
lock = threading.Lock()

def gpu_loop(gpu):
    while True:
        try:
            job = q.get_nowait()
        except queue.Empty:
            return
        try:
            rec = run_job(job, gpu)
        except Exception as e:
            rec = {"job": repr(job), "gpu": gpu, "status": "failed", "error": repr(e)}
        with lock:
            records.append(rec)
        q.task_done()

threads = [threading.Thread(target=gpu_loop, args=(g,), daemon=True) for g in [0, 1]]
for t in threads:
    t.start()
for t in threads:
    t.join()

run_manifest = pd.DataFrame(records)
display(run_manifest.sort_values(["feature", "seed"], na_position="last"))

if not run_manifest["status"].isin(["ok", "skipped_verified"]).all():
    raise RuntimeError("At least one MOLM-ST mutation job failed. Inspect the logs before continuing.")



## 6. Aggregate exactly as the revised mutation analysis

Neural seeds are first averaged **within each site**, then the primary table reports the mean and SD across the eight site blocks. This matches the revised manuscript interpretation: the eight mutation sites are the primary paired blocks; optimization seeds quantify training variability rather than biological replication.


In [ ]:

metric_frames = []
raw_frames = []
param_frames = []

for seed in SEEDS:
    for feature in FEATURE_TYPES:
        root = job_root(seed, feature)
        if not complete(seed, feature):
            raise RuntimeError(f"Missing job: seed={seed}, feature={feature}")
        metric_frames.append(pd.read_csv(root / "metrics.csv"))
        raw_frames.append(pd.read_csv(root / "raw_predictions.csv.gz"))
        param_frames.append(pd.read_csv(root / "parameter_counts.csv"))

metrics = pd.concat(metric_frames, ignore_index=True)
raw = pd.concat(raw_frames, ignore_index=True)
params = pd.concat(param_frames, ignore_index=True)

metrics.to_csv(ANALYSIS_DIR / "molm_st_mutation_metrics_by_seed.csv", index=False)
raw.to_csv(
    ANALYSIS_DIR / "molm_st_mutation_raw_predictions.csv.gz",
    index=False,
    compression="gzip"
)
params.to_csv(ANALYSIS_DIR / "molm_st_parameter_counts.csv", index=False)

site_summary = (
    metrics
    .groupby([
        "feature", "holdout_type", "analysis_role", "holdout_id",
        "kabat_site", "heldout_residue", "task", "model"
    ], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        mcc_mean=("mcc", "mean"),
        mcc_seed_sd=("mcc", "std"),
        accuracy_mean=("accuracy", "mean"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        auroc_mean=("auroc", "mean"),
        auprc_mean=("auprc", "mean"),
    )
)
site_summary.to_csv(
    ANALYSIS_DIR / "molm_st_mutation_site_metrics_mean_sd.csv",
    index=False
)

primary = site_summary[site_summary["analysis_role"] == "primary"].copy()
primary_summary = (
    primary
    .groupby(["feature", "task", "model"], as_index=False)
    .agg(
        n_sites=("holdout_id", "nunique"),
        mcc_mean=("mcc_mean", "mean"),
        mcc_site_sd=("mcc_mean", "std"),
        accuracy_mean=("accuracy_mean", "mean"),
        balanced_accuracy_mean=("balanced_accuracy_mean", "mean"),
        auroc_mean=("auroc_mean", "mean"),
        auprc_mean=("auprc_mean", "mean"),
    )
)

assert (primary_summary["n_sites"] == 8).all()

primary_summary.to_csv(
    ANALYSIS_DIR / "molm_st_primary_8site_mutation_summary.csv",
    index=False
)

display(
    primary_summary[
        primary_summary["feature"].isin(["mean_esm2", "site_esm2"])
    ].sort_values(["feature", "task"])
)

print("\nRows needed for the proposed main-table mutation block:")
for feature in ["mean_esm2", "site_esm2"]:
    for task in ["affinity", "ova"]:
        r = primary_summary[
            (primary_summary.feature == feature) &
            (primary_summary.task == task)
        ].iloc[0]
        print(
            f"{feature:10s} {task:8s}: "
            f"MCC = {r.mcc_mean:.3f} ± {r.mcc_site_sd:.3f}"
        )


## 7. Optional merge with existing four-model results

If you mount `MOLM_Unified_Experiment_Results.zip` (or an extracted compatible result dataset), this section looks for `mutation_site_metrics_mean_sd_4models.csv`, appends the dedicated MOLM-ST rows, and computes a paired **Standard-MOLM vs MOLM-ST** site-block comparison for the primary eight sites.

This merge is optional; the MOLM-ST experiment itself does not depend on the prior model-result files.


In [ ]:

from scipy.stats import wilcoxon
from itertools import product

def recursive_find_file(root, filename):
    hits = sorted(root.rglob(filename)) if root.exists() else []
    return hits[0] if hits else None

# Extract likely unified result ZIPs if mounted.
for zp in sorted(INPUT_ROOT.rglob("*.zip")) if INPUT_ROOT.exists() else []:
    if "molm_unified_experiment_results" in zp.name.lower():
        marker = EXISTING_RESULTS_DIR / (zp.stem + ".extracted")
        if not marker.exists():
            target = EXISTING_RESULTS_DIR / zp.stem
            target.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zp, "r") as zf:
                zf.extractall(target)
            marker.write_text(str(target))
            print("Extracted:", zp)

base_site = recursive_find_file(INPUT_ROOT, "mutation_site_metrics_mean_sd_4models.csv")
if base_site is None:
    base_site = recursive_find_file(EXISTING_RESULTS_DIR, "mutation_site_metrics_mean_sd_4models.csv")

def exact_sign_flip_pvalue(d):
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    if len(d) == 0:
        return np.nan
    observed = abs(d.mean())
    vals = []
    for signs in product([-1.0, 1.0], repeat=len(d)):
        vals.append(abs(np.mean(np.asarray(signs) * d)))
    return float(np.mean(np.asarray(vals) >= observed - 1e-15))

def holm_adjust(pvals):
    p = np.asarray(pvals, float)
    out = np.full(len(p), np.nan)
    valid = np.flatnonzero(np.isfinite(p))
    order = valid[np.argsort(p[valid])]
    running = 0.0
    m = len(order)
    for rank, idx in enumerate(order):
        running = max(running, (m-rank) * p[idx])
        out[idx] = min(1.0, running)
    return out

if base_site is None:
    print("No current four-model site-summary file mounted. Skipping merge and paired test.")
else:
    base = pd.read_csv(base_site)
    st = site_summary.copy()

    common = [c for c in base.columns if c in st.columns]
    five = pd.concat([base[common], st[common]], ignore_index=True)
    five.to_csv(
        ANALYSIS_DIR / "mutation_site_metrics_mean_sd_5models.csv",
        index=False
    )
    print("Merged fresh MOLM-ST with:", base_site)

    # Pair Standard-MOLM and MOLM-ST using biological/site identifiers,
    # not holdout_id strings, so the merge remains robust to naming differences.
    standard = base[
        (base["model"] == "Standard-MOLM") &
        (base["analysis_role"] == "primary")
    ].copy()
    stp = st[st["analysis_role"] == "primary"].copy()

    keys = ["feature", "task", "holdout_type", "kabat_site", "heldout_residue"]
    paired = standard[keys + ["mcc_mean"]].merge(
        stp[keys + ["mcc_mean"]],
        on=keys,
        suffixes=("_standard", "_st"),
        how="inner"
    )

    rows = []
    for (feature, task), g in paired.groupby(["feature", "task"]):
        d = g["mcc_mean_standard"].to_numpy(float) - g["mcc_mean_st"].to_numpy(float)
        if len(d) != 8:
            print("WARNING:", feature, task, "paired sites =", len(d))
        if np.allclose(d, 0):
            wstat, wp = 0.0, 1.0
        else:
            w = wilcoxon(d, zero_method="wilcox", alternative="two-sided", method="auto")
            wstat, wp = float(w.statistic), float(w.pvalue)

        rows.append({
            "feature": feature,
            "task": task,
            "n_sites": int(len(d)),
            "delta_mcc_standard_minus_st": float(d.mean()),
            "wins_standard": int((d > 0).sum()),
            "ties": int(np.isclose(d, 0).sum()),
            "losses_standard": int((d < 0).sum()),
            "wilcoxon_statistic": wstat,
            "wilcoxon_p": wp,
            "exact_sign_flip_p": exact_sign_flip_pvalue(d),
        })

    paired_tests = pd.DataFrame(rows)
    paired_tests["wilcoxon_p_holm"] = holm_adjust(paired_tests["wilcoxon_p"])
    paired_tests.to_csv(
        ANALYSIS_DIR / "standard_vs_molm_st_primary_siteblock_tests.csv",
        index=False
    )
    display(paired_tests.sort_values(["feature", "task"]))



## 8. Reproducibility manifest and result ZIP


In [ ]:

from datetime import datetime, timezone

manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "repository_url": REPO_URL,
    "repository_commit": PINNED_COMMIT,
    "model": "original plain MOLM-ST",
    "implementation": "phase0_config.train_molm_st",
    "not_a_h_arm_f": True,
    "seeds": SEEDS,
    "features": FEATURE_DIMS,
    "holdout_protocol": {
        "primary": "8 top-residue grouped mutation holdouts",
        "secondary_wildtype_enabled": bool(RUN_WILDTYPE_SECONDARY),
        "sites_kabat": list(map(int, cfg.MUTATION_SITES_KABAT)),
        "python_indices": list(map(int, cfg.MUTATION_SITES)),
    },
    "training": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "shared_dims": [256, 128],
        "tower_dims": [64, 32],
        "latent_dim": 16,
        "dropout": 0.2,
        "target_ranking_weight": 0.3,
        "ova_ranking_weight": 0.6,
        "target_gap_weight": 0.2,
        "ova_gap_weight": 0.6,
        "ranking_margin": 0.3,
        "gap_margin": 0.2,
        "target_seed_offset": 500,
        "ova_seed_offset": 600,
        "pareto_loss": False,
        "adversarial_weight": 0.0,
        "orthogonality_weight": 0.0,
    },
    "aggregation": (
        "average five optimization seeds within each holdout site, "
        "then report mean±SD across the eight primary sites"
    ),
}

(ANALYSIS_DIR / "MOLM_ST_MUTATION_REPRODUCIBILITY_MANIFEST.json").write_text(
    json.dumps(manifest, indent=2)
)

bundle = WORK_ROOT / "bundle"
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

for p in ANALYSIS_DIR.iterdir():
    if p.is_file():
        shutil.copy2(p, bundle / p.name)

shutil.copy2(HOLDOUT_PATH, bundle / HOLDOUT_PATH.name)
shutil.copy2(WORKER_PATH, bundle / WORKER_PATH.name)

zip_base = Path("/kaggle/working/MOLM_ST_MutationHoldout_Results")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=bundle)

print("RESULT ZIP:", zip_path)
print("\nKey manuscript file:")
print(ANALYSIS_DIR / "molm_st_primary_8site_mutation_summary.csv")
print("\nIf the current four-model result bundle was mounted, also inspect:")
print(ANALYSIS_DIR / "standard_vs_molm_st_primary_siteblock_tests.csv")
